In [0]:
tier = "silver"
storage_account = "databrickslrd"

spark.sql("CREATE SCHEMA IF NOT EXISTS " + tier)
spark.sql("USE " + tier)

access_key = dbutils.secrets.get(
    scope="olist-scope",
    key="access-key"
)


spark.conf.set(
    f"fs.azure.account.key.{storage_account}.blob.core.windows.net",
    access_key
) 



In [0]:
%sql
DROP TABLE IF EXISTS olist_customers;
CREATE TABLE olist_customers 
AS
SELECT 
  customer_id as ID_CUSTOMER,
  customer_unique_id AS ID_CUSTOMER_UNIQUE,
  customer_zip_code_prefix AS ZIP_CODE_PREFIX,
  UPPER(customer_city) AS CITY,
  UPPER(customer_state) AS STATE
FROM bronze.customers_raw


In [0]:
df = spark.table("olist_customers")

df.write.format("delta").mode("overwrite").save(f"wasbs://{tier}@{storage_account}.blob.core.windows.net/olist_customers")

In [0]:
%sql
DROP TABLE IF EXISTS olist_geolocation;

CREATE TABLE olist_geolocation
AS
SELECT 
  geolocation_zip_code_prefix AS ZIP_CODE_PREFIX,
  geolocation_lat AS LATITUDE,
  geolocation_lng AS LONGITUDE,
  UPPER(geolocation_city) AS CITY,
  geolocation_state AS STATE
FROM bronze.geolocation_raw

In [0]:
df = spark.table("olist_geolocation")

df.write.format("delta").mode("overwrite").save(f"wasbs://{tier}@{storage_account}.blob.core.windows.net/olist_geolocation")

In [0]:
%sql
DROP TABLE IF EXISTS olist_order_items;

CREATE TABLE olist_order_items
AS
SELECT 
  order_id AS ID_ORDER,
  order_item_id AS ID_ORDER_ITEM,
  product_id AS ID_PRODUCT,
  price AS UNIT_PRICE,
  freight_value AS UNIT_FREIGHT
FROM bronze.order_items_raw

In [0]:
df = spark.table("olist_order_items")

df.write.format("delta").mode("overwrite").save(f"wasbs://{tier}@{storage_account}.blob.core.windows.net/olist_order_items")

In [0]:
%sql

DROP TABLE IF EXISTS olist_order_payments;

CREATE TABLE olist_order_payments
AS
SELECT 
  order_id AS ID_ORDER,
  payment_sequential AS ID_SEQUENTIAL,
  payment_type AS PAYMENT_TYPE,
  payment_installments AS PAYMENT_INSTALLMENTS,
  payment_value AS PAYMENT_VALUE
FROM bronze.order_payments_raw
WHERE payment_type != 'not_defined'

In [0]:
df = spark.table("olist_order_payments")

df.write.format("delta").mode("overwrite").save(f"wasbs://{tier}@{storage_account}.blob.core.windows.net/olist_order_payments")


In [0]:
%sql
DROP TABLE IF EXISTS olist_order_reviews;

CREATE TABLE olist_order_reviews
AS
SELECT 
  review_id AS ID_REVIEW,
  order_id AS ID_ORDER,
  review_score AS REVIEW_SCORE,
  TRIM(LOWER(review_comment_title)) AS REVIEW_COMMENT,
  TRIM(LOWER(review_comment_message)) AS REVIEW_MESSAGE,
  review_creation_date AS REVIEW_CREATION_DATE,
  review_answer_timestamp AS REVIEW_ANSWER
FROM bronze.order_reviews_raw
WHERE length(REVIEW_SCORE) = 1 

In [0]:
df = spark.table("olist_order_reviews")

df.write.format("delta").mode("overwrite").save(f"wasbs://{tier}@{storage_account}.blob.core.windows.net/olist_order_reviews")


In [0]:
%sql
DROP TABLE IF EXISTS olist_orders;

CREATE TABLE olist_orders
AS
SELECT 
  order_id AS ID_ORDER,
  customer_id AS ID_CUSTOMER,
  order_status AS ORDER_STATUS,
  order_purchase_timestamp AS ORDER_PURCHASE_DATE,
  order_approved_at AS ORDER_APPROVED_AT,
  order_delivered_carrier_date AS ORDER_DELIVERED_CARRIER_DATE,
  order_delivered_customer_date AS ORDER_DELIVERED_CUSTOMER_DATE,
  order_estimated_delivery_date AS ORDER_ESTIMATED_DELIVERY_DATE
FROM bronze.orders_raw

In [0]:
df = spark.table("olist_orders")

df.write.format("delta").mode("overwrite").save(f"wasbs://{tier}@{storage_account}.blob.core.windows.net/olist_orders")

In [0]:
%sql
DROP TABLE IF EXISTS olist_products;

CREATE TABLE olist_products
AS
SELECT 
  product_id AS ID_PRODUCT,
  product_category_name AS PRODUCT_CATEGORY,
  product_name_lenght AS PRODUCT_NAME_LENGTH,
  product_description_lenght AS PRODUCT_DESCRIPTION_LENGTH,
  product_photos_qty AS PRODUCT_PHOTOS_QTY,
  product_weight_g AS PRODUCT_WEIGHT,
  product_length_cm AS PRODUCT_LENGTH,
  product_height_cm AS PRODUCT_HEIGHT,
  product_width_cm AS PRODUCT_WIDTH
FROM bronze.products_raw

In [0]:
df = spark.table("olist_products")

df.write.format("delta").mode("overwrite").save(f"wasbs://{tier}@{storage_account}.blob.core.windows.net/olist_products")


In [0]:
%sql
DROP TABLE IF EXISTS olist_sellers;

CREATE TABLE olist_sellers
AS
SELECT 
  seller_id AS ID_SELLER,
  id_
  seller_zip_code_prefix AS ZIP_CODE_PREFIX,
  UPPER(seller_city) AS CITY,
  seller_state AS STATE
FROM bronze.sellers_raw

In [0]:
df = spark.table("olist_sellers")

df.write.format("delta").mode("overwrite").save(f"wasbs://{tier}@{storage_account}.blob.core.windows.net/olist_sellers")